In [1]:
import csv
from itertools import product

# ============================================================
# Define the candidate metal elements
# ============================================================
# metals = ["Fe", "Co", "Ni", "Mn", "Cu", "Zn", "Mo", "Ru"]
metals = [
    "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Zr", "Nb", "Mo", "Ru", "Pd", "Ag", "Cd",
    "Ta", "W", "Ir", "Pt", "La", "Ce",
    "Al", "In", "Sn"
]

# Templates for monometallic compounds
nitride_templates = ["MN", "M2N", "M3N2"]
sulfide_templates = ["MS", "MS2", "M2S3"]
oxide_templates = ["MO", "M2O3", "MO2", "M3O4"]

# Store the generated monometallic compounds
single_metal_compounds = []

# ============================================================
# Generate monometallic compounds
# ============================================================
for metal in metals:
    # Generate nitrides
    for template in nitride_templates:
        compound = template.replace("M", metal)
        single_metal_compounds.append(
            {
                "Metal": metal,
                "Compound": compound,
                "Type": "Nitride"
            }
        )

    # Generate sulfides
    for template in sulfide_templates:
        compound = template.replace("M", metal)
        single_metal_compounds.append(
            {
                "Metal": metal,
                "Compound": compound,
                "Type": "Sulfide"
            }
        )

    # Generate oxides
    for template in oxide_templates:
        compound = template.replace("M", metal)
        single_metal_compounds.append(
            {
                "Metal": metal,
                "Compound": compound,
                "Type": "Oxide"
            }
        )

# print(single_metal_compounds)

# ============================================================
# Save the generated compounds
# ============================================================
output_file = "single_metal_compounds.csv"

with open(output_file, "w", newline="") as csvfile:
    fieldnames = ["Metal", "Compound", "Type"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in single_metal_compounds:
        writer.writerow(row)

print(
    f"Generated {len(single_metal_compounds)} monometallic compounds "
    f"and saved them to {output_file}."
)

Generated 250 monometallic compounds and saved them to single_metal_compounds.csv.


In [2]:
import itertools
import re

import pandas as pd

# ============================================================
# 1. Define the metal elements used for five-metal compounds
# ============================================================
metals = [
    "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Ru", "Pd", "Ag", "Mo", "In"
]

# ============================================================
# 2. Load the monometallic parent compounds
# ============================================================
df_single = pd.read_csv("single_metal_compounds.csv")

# Extract parent formulas from the Compound column
single_metal_compounds = df_single["Compound"].tolist()

# ============================================================
# 3. Define the main-metal fractions
# ============================================================
MAIN_RATIOS = [0.5, 0.6]

# ============================================================
# 4. Define the auxiliary-metal fraction templates
# ============================================================

# Main-metal fraction = 0.5; total auxiliary-metal fraction = 0.5
TEMPLATES_05 = [
    [0.2, 0.1, 0.1, 0.1],
    # [0.15, 0.15, 0.1, 0.1],
    [0.2, 0.15, 0.1, 0.05],
    # [0.15, 0.15, 0.15, 0.05],
    [0.2, 0.2, 0.05, 0.05]
]

# Main-metal fraction = 0.6; total auxiliary-metal fraction = 0.4
TEMPLATES_06 = [
    [0.1, 0.1, 0.1, 0.1],
    [0.15, 0.1, 0.1, 0.05],
    [0.2, 0.1, 0.05, 0.05]
]

TEMPLATE_DICT = {
    0.5: TEMPLATES_05,
    0.6: TEMPLATES_06
}


# ============================================================
# Extract the stoichiometric coefficient of the parent metal
# Example: Co3O4 -> 3; NiO -> 1
# ============================================================
def count_base_metal(formula):
    match = re.match(r"([A-Z][a-z]?)(\d*)", formula)

    if match:
        num = match.group(2)
        return int(num) if num else 1

    return 1


# ============================================================
# Generate five-metal compounds
# ============================================================
def generate_five_metal_compounds():
    results = []

    for base in single_metal_compounds:
        base_m = count_base_metal(base)

        # Extract the main-metal symbol from the parent formula
        # Example: Co3O4 -> Co
        main_metal_symbol = re.match(
            r"([A-Z][a-z]?)",
            base
        ).group(1)

        for main_ratio in MAIN_RATIOS:
            templates = TEMPLATE_DICT[main_ratio]

            # Exclude the main metal from the auxiliary-metal pool
            other_metals = [
                metal
                for metal in metals
                if metal != main_metal_symbol
            ]

            # Select four auxiliary metals
            for four in itertools.combinations(other_metals, 4):
                for temp in templates:
                    # Combine auxiliary-metal and main-metal fractions
                    ratios = temp + [main_ratio]

                    # Scale the fractions according to the parent formula
                    # Example: Co3O4 uses a scaling factor of 3
                    scaled_ratios = [
                        round(ratio * base_m, 4)
                        for ratio in ratios
                    ]

                    # Construct the formula with the main metal placed last
                    formula = ""

                    for metal, ratio in zip(
                        four + (main_metal_symbol,),
                        scaled_ratios
                    ):
                        formula += f"{metal}{ratio}"

                    # Append the remaining part of the parent formula,
                    # such as O4, S2, or N
                    suffix = base[len(main_metal_symbol):]
                    formula += suffix

                    results.append(
                        {
                            "base": base,
                            "metals": four + (main_metal_symbol,),
                            "ratios": scaled_ratios,
                            "compound": formula
                        }
                    )

    return pd.DataFrame(results)


# ============================================================
# Generate and save the five-metal compounds
# ============================================================
df = generate_five_metal_compounds()

df.to_csv(
    "five_metal_compounds.csv",
    index=False
)

df.head()

,base,metals,ratios,compound
0,TiN,"(Mn, Fe, Co, Ni, Ti)","[0.2, 0.1, 0.1, 0.1, 0.5]",Mn0.2Fe0.1Co0.1Ni0.1Ti0.5N
1,TiN,"(Mn, Fe, Co, Ni, Ti)","[0.2, 0.15, 0.1, 0.05, 0.5]",Mn0.2Fe0.15Co0.1Ni0.05Ti0.5N
2,TiN,"(Mn, Fe, Co, Ni, Ti)","[0.2, 0.2, 0.05, 0.05, 0.5]",Mn0.2Fe0.2Co0.05Ni0.05Ti0.5N
3,TiN,"(Mn, Fe, Co, Cu, Ti)","[0.2, 0.1, 0.1, 0.1, 0.5]",Mn0.2Fe0.1Co0.1Cu0.1Ti0.5N
4,TiN,"(Mn, Fe, Co, Cu, Ti)","[0.2, 0.15, 0.1, 0.05, 0.5]",Mn0.2Fe0.15Co0.1Cu0.05Ti0.5N


In [3]:
print("Total:", len(df))

Total: 415800


In [5]:
#做随机抽样，在生成的所有五金属组合里面随机抽取一万条氮化物，一万条氧化物，一万条硫化物
import itertools
import pandas as pd
import re
import random

# ============================================================
# 读取单金属化合物
# ============================================================
df_single = pd.read_csv("single_metal_compounds.csv")

# 按 Type 分类
groups = {
    "Oxide": df_single[df_single["Type"] == "Oxide"],
    "Sulfide": df_single[df_single["Type"] == "Sulfide"],
    "Nitride": df_single[df_single["Type"] == "Nitride"]
}

# 单金属对应金属列表
# metals = sorted(df_single["Metal"].unique())
metals = ["Mn","Fe","Co","Ni","Cu","Zn","Ru","Pd","Ag","Mo","In"]

# ============================================================
# 主金属摩尔比
# ============================================================
MAIN_RATIOS = [0.5, 0.6]

TEMPLATES_05 = [
    [0.2, 0.1, 0.1, 0.1],
    [0.2, 0.15, 0.1, 0.05],
    [0.2, 0.2, 0.05, 0.05]
]

TEMPLATES_06 = [
    [0.1, 0.1, 0.1, 0.1],
    [0.15, 0.1, 0.1, 0.05],
    [0.2, 0.1, 0.05, 0.05]
]

TEMPLATE_DICT = {0.5: TEMPLATES_05, 0.6: TEMPLATES_06}


# ============================================================
# 工具函数：解析母相主金属数量，如 Co3O4 → 3
# ============================================================
def count_base_metal(formula):
    match = re.match(r"([A-Z][a-z]?)(\d*)", formula)
    if not match:
        return 1
    num = match.group(2)
    return int(num) if num else 1


# ============================================================
# 核心函数：为某一类生成五金属
# target_count 约为 10,000
# ============================================================
def generate_for_category(df_cat, target_count=10000):
    results = []

    for _, row in df_cat.iterrows():
        base = row["Compound"]
        main_metal_symbol = row["Metal"]
        base_m = count_base_metal(base)

        # 其他金属列表（不能包含主金属）
        other_metals = [m for m in metals if m != main_metal_symbol]

        for main_ratio in MAIN_RATIOS:
            templates = TEMPLATE_DICT[main_ratio]

            # 所有辅金属组合
            combinations_4 = itertools.combinations(other_metals, 4)

            for four in combinations_4:
                for temp in templates:

                    ratios = temp + [main_ratio]
                    scaled = [r * base_m for r in ratios]

                    # 构造化学式（主金属放最后）
                    formula = ""
                    for m, r in zip(four + (main_metal_symbol,), scaled):
                        formula += f"{m}{round(r,3)}"

                    # 接 suffix (例如 3O4)
                    suffix = base[len(main_metal_symbol):]
                    formula += suffix

                    results.append({
                        "base": base,
                        "metals": four + (main_metal_symbol,),
                        "ratios": scaled,
                        "compound": formula
                    })

    # 随机抽样约 10000 条
    if len(results) > target_count:
        results = random.sample(results, target_count)

    return pd.DataFrame(results)


# ============================================================
# 执行生成
# ============================================================
output_files = {
    "Oxide": "five_metal_oxides.csv",
    "Sulfide": "five_metal_sulfides.csv",
    "Nitride": "five_metal_nitrides.csv"
}

for cat, df_cat in groups.items():
    df_res = generate_for_category(df_cat, target_count=10000)
    df_res.to_csv(output_files[cat], index=False)
    print(f"{cat} → {len(df_res)} 条，已保存到 {output_files[cat]}")

Oxide → 10000 条，已保存到 five_metal_oxides.csv
Sulfide → 10000 条，已保存到 five_metal_sulfides.csv
Nitride → 10000 条，已保存到 five_metal_nitrides.csv


In [2]:
import itertools
import pandas as pd
import re

# =====================================
# 1. 单金属生成
# =====================================
metals_all = ['Ti', 'V','Cr','Mn','Fe','Co','Ni','Cu','Zn',
         'Zr','Nb','Mo','Ru','Pd','Ag','Cd',
         'Ta','W','Ir','Pt','La','Ce',
         'Al','In','Sn']

nitride_templates = ['MN', 'M2N', 'M3N2']
sulfide_templates = ['MS', 'MS2', 'M2S3']
oxide_templates = ['MO', 'M2O3', 'MO2', 'M3O4']

single_metal_compounds = []

for metal in metals_all:
    for t in nitride_templates:
        single_metal_compounds.append(t.replace('M', metal))
    for t in sulfide_templates:
        single_metal_compounds.append(t.replace('M', metal))
    for t in oxide_templates:
        single_metal_compounds.append(t.replace('M', metal))

print(f"单金属化合物数量: {len(single_metal_compounds)}")

# =====================================
# 2. 五金属参数
# =====================================
metals = ["Mn","Fe","Co","Ni","Cu","Zn","Ru","Pd","Ag","Mo","In"]

MAIN_RATIOS = [0.5, 0.6]

TEMPLATES_05 = [
    [0.2, 0.1, 0.1, 0.1],
    [0.2, 0.15, 0.1, 0.05],
    [0.2, 0.2, 0.05, 0.05]
]

TEMPLATES_06 = [
    [0.1, 0.1, 0.1, 0.1],
    [0.15, 0.1, 0.1, 0.05],
    [0.2, 0.1, 0.05, 0.05]
]

TEMPLATE_DICT = {
    0.5: TEMPLATES_05,
    0.6: TEMPLATES_06
}

# =====================================
# 3. 工具函数
# =====================================
def count_base_metal(formula):
    match = re.match(r"([A-Z][a-z]?)(\d*)", formula)
    if match:
        num = match.group(2)
        return int(num) if num else 1
    return 1

def canonical_key(metals, ratios):
    return tuple(sorted(zip(metals, ratios)))

def format_formula(all_metals, ratios, main_metal, suffix):
    pairs = list(zip(all_metals, ratios))
    main_pair = [p for p in pairs if p[0] == main_metal]
    others = [p for p in pairs if p[0] != main_metal]
    others_sorted = sorted(others)
    final_pairs = others_sorted + main_pair
    formula = ""
    for m, r in final_pairs:
        formula += f"{m}{r}"
    return formula + suffix

# =====================================
# 4. 五金属生成（任意比例排列）
# =====================================
def generate_five_metal_compounds_permutation():
    results = []
    seen = set()

    for base in single_metal_compounds:

        base_m = count_base_metal(base)
        main_metal = re.match(r"([A-Z][a-z]?)", base).group(1)

        for main_ratio in MAIN_RATIOS:
            templates = TEMPLATE_DICT[main_ratio]

            other_metals = [m for m in metals if m != main_metal]

            for four in itertools.combinations(other_metals, 4):

                for temp in templates:
                    # 对四个金属比例生成所有排列
                    for perm_ratio in itertools.permutations(temp):
                        ratios = list(perm_ratio) + [main_ratio]
                        scaled = [round(r * base_m, 4) for r in ratios]

                        all_metals = list(four) + [main_metal]

                        # 去重：组合 + 比例
                        key = (base, canonical_key(all_metals, scaled))
                        if key in seen:
                            continue
                        seen.add(key)

                        suffix = base[len(main_metal):]

                        formula = format_formula(
                            all_metals,
                            scaled,
                            main_metal,
                            suffix
                        )

                        results.append({
                            "base": base,
                            "compound": formula
                        })

    return pd.DataFrame(results)

# =====================================
# 5. 运行
# =====================================
df = generate_five_metal_compounds_permutation()
df.to_csv("five_metal_compounds_permutation.csv", index=False)
print(f"五金属化合物数量（任意比例排列）: {len(df)}")
print(df.head())

单金属化合物数量: 250
五金属化合物数量（任意比例排列）: 4088700
  base                      compound
0  TiN    Co0.1Fe0.1Mn0.2Ni0.1Ti0.5N
1  TiN    Co0.1Fe0.2Mn0.1Ni0.1Ti0.5N
2  TiN    Co0.2Fe0.1Mn0.1Ni0.1Ti0.5N
3  TiN    Co0.1Fe0.1Mn0.1Ni0.2Ti0.5N
4  TiN  Co0.1Fe0.15Mn0.2Ni0.05Ti0.5N


In [4]:
import itertools
import pandas as pd
import re

# ===========================
# 1. 单金属生成
# ===========================
metals_all = ['Ti', 'V','Cr','Mn','Fe','Co','Ni','Cu','Zn',
              'Zr','Nb','Mo','Ru','Pd','Ag','Cd',
              'Ta','W','Ir','Pt','La','Ce',
              'Al','In','Sn']

nitride_templates = ['MN', 'M2N', 'M3N2']
sulfide_templates = ['MS', 'MS2', 'M2S3']
oxide_templates = ['MO', 'M2O3', 'MO2', 'M3O4']

single_metal_compounds = []
for metal in metals_all:
    for t in nitride_templates:
        single_metal_compounds.append(t.replace('M', metal))
    for t in sulfide_templates:
        single_metal_compounds.append(t.replace('M', metal))
    for t in oxide_templates:
        single_metal_compounds.append(t.replace('M', metal))

# ===========================
# 2. 五金属参数
# ===========================
metals = ["Mn","Fe","Co","Ni","Cu","Zn","Ru","Pd","Ag","Mo","In"]
MAIN_RATIOS = [0.5, 0.6]
TEMPLATES_05 = [[0.2,0.1,0.1,0.1],[0.2,0.15,0.1,0.05],[0.2,0.2,0.05,0.05]]
TEMPLATES_06 = [[0.1,0.1,0.1,0.1],[0.15,0.1,0.1,0.05],[0.2,0.1,0.05,0.05]]
TEMPLATE_DICT = {0.5:TEMPLATES_05,0.6:TEMPLATES_06}

# ===========================
# 3. 工具函数
# ===========================
def count_base_metal(formula):
    match = re.match(r"([A-Z][a-z]?)(\d*)", formula)
    if match:
        num = match.group(2)
        return int(num) if num else 1
    return 1

def format_formula(all_metals, ratios, main_metal, suffix):
    pairs = list(zip(all_metals, ratios))
    main_pair = [p for p in pairs if p[0]==main_metal]
    others = [p for p in pairs if p[0]!=main_metal]
    others_sorted = sorted(others)
    final_pairs = others_sorted + main_pair
    formula = "".join(f"{m}{r}" for m,r in final_pairs)
    return formula + suffix

def canonical_key(metals, ratios):
    return tuple(sorted(zip(metals, ratios)))

# ===========================
# 4. 折中生成函数
# ===========================
def generate_five_metal_compounds_flexible():
    results = []
    seen = set()

    for base in single_metal_compounds:
        base_m = count_base_metal(base)
        main_metal = re.match(r"([A-Z][a-z]?)", base).group(1)

        for main_ratio in MAIN_RATIOS:
            templates = TEMPLATE_DICT[main_ratio]
            other_metals = [m for m in metals if m != main_metal]

            # 4金属组合
            for four in itertools.combinations(other_metals, 4):
                # 对四金属生成排列组合，但只考虑模板比例匹配的组合（避免生成重复排列）
                for temp in templates:
                    # 生成四金属所有比例排列
                    for perm_ratio in set(itertools.permutations(temp)):
                        ratios = list(perm_ratio) + [main_ratio]
                        scaled = [round(r*base_m,4) for r in ratios]
                        all_metals = list(four) + [main_metal]

                        key = (base, canonical_key(all_metals, scaled))
                        if key in seen:
                            continue
                        seen.add(key)

                        suffix = base[len(main_metal):]
                        formula = format_formula(all_metals, scaled, main_metal, suffix)

                        results.append({"base": base, "compound": formula})

    return pd.DataFrame(results)

# ===========================
# 5. 运行
# ===========================
df = generate_five_metal_compounds_flexible()
df.to_csv("five_metal_compounds_flexible.csv", index=False)
print(f"五金属化合物数量（灵活版）: {len(df)}")
print(df.head())

五金属化合物数量（灵活版）: 4088700
  base                      compound
0  TiN    Co0.2Fe0.1Mn0.1Ni0.1Ti0.5N
1  TiN    Co0.1Fe0.2Mn0.1Ni0.1Ti0.5N
2  TiN    Co0.1Fe0.1Mn0.2Ni0.1Ti0.5N
3  TiN    Co0.1Fe0.1Mn0.1Ni0.2Ti0.5N
4  TiN  Co0.15Fe0.1Mn0.05Ni0.2Ti0.5N
